In [1]:
# Load tests from results folder
import pickle, os
from tqdm import tqdm

def load_tests(symbol, conf):
    folder = f'/_data/base/{symbol}/results/{conf}'
    assert os.path.isdir(folder), f'Results folder not found: {folder}'

    files = [fn for fn in os.listdir(folder) if fn.endswith('.pkl')]
    files.sort()
    assert len(files) > 0, f'No result files in: {folder}'

    sizes_mb = 0.0
    count = 0

    test_data: list = []
    pbar = tqdm(total=len(files), desc=f'Loading tests for {symbol} {conf}')

    for fn in files:
        pbar.update(1)
        path = f'{folder}/{fn}'
        assert os.path.isfile(path), f'Not a file: {path}'
        size_mb = os.path.getsize(path) / 1024 / 1024
        with open(path, 'rb') as f:
            data = pickle.load(f)
        assert isinstance(data, list), f'File is not a list: {path}'
        test_data.extend(data)
        sizes_mb += size_mb
        count += len(data)
    
    pbar.close()

    print(f'Total size: {sizes_mb:.3f} MB')
    print(f'Loaded total: {len(test_data)} tests from {len(files)} files')

    return test_data

In [2]:
# Load confs
import json

def load_confs():
    folder = './run_confs'
    confs = {}

    for file in os.listdir(folder):
        if not file.endswith('.json'):
            continue

        with open(f'{folder}/{file}', 'r') as f:
            conf = json.load(f)
            confs[file.replace('.json', '')] = conf

    return confs

In [3]:
# Base parameters for cost regression
import numpy as np

def get_base_metrics(params, volume_usd):
    """
    Метрики:
    - average_pnl_pct: средняя прибыль в процентах
    - average_position_duration_s: средняя длительность позиции в секундах
    - positions_per_year: количество позиций в год
    - cost_per_year_pct: стоимость в процентах в год
    - cost_per_year_usd: стоимость в USD в год
    - avg_chases_per_position: среднее количество чейзов в позиции
    - avg_turnover_per_usd: средний оборот на доллар в год
    - avg_turnover_per_year: средний оборот в год
    """
    metrics = {}

    # Average PnL% per position
    base_average_pnl_pct = np.mean(params[:, 0])
    metrics['average_pnl_pct'] = base_average_pnl_pct

    # Average position duration (seconds)
    base_average_position_duration_s = np.mean(params[:, 1])
    metrics['average_position_duration_s'] = base_average_position_duration_s

    # Positions per year
    base_positions_per_year = 365*24*60*60.0 / base_average_position_duration_s
    metrics['positions_per_year'] = base_positions_per_year

    # Cost per year
    base_cost_per_year_pct = base_average_pnl_pct * 365*24*60*60.0 / base_average_position_duration_s
    metrics['cost_per_year_pct'] = base_cost_per_year_pct

    # Cost per year in USD
    base_cost_per_year_usd = base_cost_per_year_pct * volume_usd / 100
    metrics['cost_per_year_usd'] = base_cost_per_year_usd

    # Average chases per position
    base_avg_chases_per_position = np.mean(params[:, 2])
    metrics['avg_chases_per_position'] = base_avg_chases_per_position

    # Annual turnover per dollar per year
    base_avg_turnover_per_usd = base_positions_per_year * base_avg_chases_per_position
    metrics['avg_turnover_per_usd'] = base_avg_turnover_per_usd

    # And in USDT
    base_avg_turnover_per_year = base_avg_turnover_per_usd * volume_usd
    metrics['avg_turnover_per_year'] = base_avg_turnover_per_year

    return metrics

def base_report(test_data, symbol, conf):
    """
    Концепция базовых показателей: идет расчет стоимости полного покрытия
    (по времени) длительностью 1 год. Расчет является регрессией на основе
    данных истории и бэктеста.
    """

    # Checks
    assert isinstance(conf['test_volume_usd'], list), 'non-discrete test_volume_usd not supported'
   
    # Gather params
    all_params = {}

    for item in test_data:
        total_trades_executed = int(item['stat']['agg']['total_trades_executed'])
        if total_trades_executed == 0:
            continue
        
        # Parametrization
        test_volume_usd = int(item['run_conf']['test_volume_usd'])
        only_side = int(item['run_conf']['only_side'])
        
        # Metrics
        full_execution_time = float(item['stat']['agg']['full_execution_time'])
        pnl = float(item['stat']['agg']['pnl'])
        n_chases = int(item['stat']['agg']['n_chases'])

        # Add
        if only_side not in all_params:
            all_params[only_side] = {}

        if test_volume_usd not in all_params[only_side]:
            all_params[only_side][test_volume_usd] = []
        
        all_params[only_side][test_volume_usd].append((pnl / test_volume_usd * 100.0, full_execution_time, n_chases))

    # Generate report
    csv = []
    
    for only_side in sorted(all_params):
        for test_volume_usd in sorted(all_params[only_side]):
            # Convert to numpy array
            params = np.array(all_params[only_side][test_volume_usd], dtype=np.float64)
            if len(params) == 0:
                continue

            # Get metrics
            metrics = get_base_metrics(params, test_volume_usd)

            # Add to CSV
            csv.append({
                'symbol': symbol,
                'conf': conf['description'],
                'only_side': only_side,
                'test_volume_usd': test_volume_usd,
                **metrics
            })

    return csv


In [4]:
# Generate report
import os, pandas as pd
from itables import show

DIR = '/_data/base'

# List all symbols
symbols = [fn for fn in os.listdir(DIR) if os.path.isdir(f'{DIR}/{fn}')]
symbols.sort()
print(f'Symbols: {symbols}')

csv = []

for symbol in symbols:
    print(f'Building reports for: {symbol}')

    # Load confs
    folder = f'/_data/base/{symbol}/results'
    confs = load_confs()
    print(f'Found {len(confs)} configs')

    for name, conf in confs.items():
        print(f'Building report for: {symbol} {name}')

        # Load tests
        test_data = load_tests(symbol, name)

        # Build report
        csv.extend(base_report(test_data, symbol, conf))

# Convert to DataFrame
df = pd.DataFrame(csv)

# Show
show(df)

# Save
os.makedirs('reports', exist_ok=True)
df.to_csv('reports/base_report.csv', index=False)


Symbols: ['BTCUSDC', 'ETHUSDC', 'SOLUSDC']
Building reports for: BTCUSDC
Found 4 configs
Building report for: BTCUSDC list_01p


Loading tests for BTCUSDC list_01p: 100%|██████████| 100/100 [00:16<00:00,  6.04it/s]


Total size: 921.409 MB
Loaded total: 974993 tests from 100 files
Building report for: BTCUSDC list_01p_s


Loading tests for BTCUSDC list_01p_s: 100%|██████████| 100/100 [00:12<00:00,  8.19it/s]


Total size: 267.403 MB
Loaded total: 999917 tests from 100 files
Building report for: BTCUSDC list_1u_s


...

Loading tests for SOLUSDC list_1u_s: 100%|██████████| 100/100 [00:09<00:00, 10.03it/s]


Total size: 263.554 MB
Loaded total: 999758 tests from 100 files
Building report for: SOLUSDC list_1u


Loading tests for SOLUSDC list_1u: 100%|██████████| 100/100 [00:15<00:00,  6.66it/s]


Total size: 563.128 MB
Loaded total: 995988 tests from 100 files


Loading ITables v2.6.2 from the internet... (need help?)
